[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/068.%20The%20AS_RS%20Task%20Interleaving%20Problem/P68-Tier-6_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 68. The AS/RS Task Interleaving Problem
## Tier 6: Autonomous & Self-Optimizing Ecosystem (Multi-Agent System)

### Key Assumptions
- Distributed intelligence with multiple autonomous agents (crane agents, task agents, coordinator agent)
- Contract Net Protocol for agent negotiations and task allocation
- BDI (Belief-Desire-Intention) agent architecture with message passing
- Emergent behavior through local agent interactions without central control
- Market-based resource allocation with dynamic pricing mechanisms
- Self-organizing system that adapts to changing conditions

### Approach (Step-by-Step)
1. **Agent Architecture**: Design BDI agents with beliefs, desires, and intentions
2. **Communication Protocol**: Implement Contract Net Protocol for task allocation
3. **Market Mechanism**: Create dynamic pricing for task bidding
4. **Coordination Logic**: Develop distributed decision-making algorithms
5. **Emergent Behavior**: Analyze system-level properties from local interactions

### What to Look for in the Results
- Task allocation efficiency through agent negotiations
- System performance under varying load conditions
- Emergent behavior patterns and self-organization
- Comparison with centralized optimization approaches
- Scalability analysis with different numbers of agents
- Resilience to agent failures and communication disruptions

### Concrete Example (from the source)
Multi-agent system with 3 crane agents, task agents, and coordinator:

**Agent Configuration**:
- Crane Agents: 3 autonomous agents with local decision-making
- Task Agents: Dynamic agents representing storage/retrieval requests
- Coordinator Agent: Facilitates agent communication and resolves conflicts

**Expected Performance**:
- Task allocation efficiency: 92% (vs 85% centralized)
- System throughput: 187 tasks/hour (vs 156 sequential)
- Adaptation time: 3.2 seconds for new task patterns
- Failure resilience: 78% performance with 1 agent failure
- Communication overhead: 12% of total processing time

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries for Multi-Agent System
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import time
from collections import defaultdict, deque
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional, Any
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# Set up professional visualization style
plt.style.use('default')
sns.set_palette("husl")

In [ ]:
class AgentType(Enum):
    """Types of agents in the multi-agent system."""
    CRANE = "crane"
    TASK = "task"
    COORDINATOR = "coordinator"
    MONITOR = "monitor"

@dataclass
class Message:
    """Communication message between agents."""
    sender: str
    receiver: str
    message_type: str
    content: Dict[str, Any]
    timestamp: float
    priority: int = 1

@dataclass
class Task:
    """Task representation for multi-agent system."""
    id: str
    task_type: str  # 'storage' or 'retrieval'
    x: int
    y: int
    priority: int
    deadline: Optional[float] = None
    created_time: float = 0.0
    assigned_agent: Optional[str] = None
    completed_time: Optional[float] = None
    bids: List[Dict[str, Any]] = field(default_factory=list)

@dataclass
class AgentBeliefs:
    """Beliefs of an agent about the environment."""
    agent_positions: Dict[str, Tuple[int, int]] = field(default_factory=dict)
    task_queue: List[Task] = field(default_factory=list)
    system_load: float = 0.0
    market_prices: Dict[str, float] = field(default_factory=dict)
    agent_capabilities: Dict[str, Dict[str, float]] = field(default_factory=dict)

@dataclass
class AgentDesires:
    """Desires or goals of an agent."""
    maximize_utility: bool = True
    minimize_completion_time: bool = True
    maintain_workload_balance: bool = True
    ensure_deadline_compliance: bool = True
    target_utilization: float = 0.8

@dataclass
class AgentIntentions:
    """Intentions or planned actions of an agent."""
    current_tasks: List[Task] = field(default_factory=list)
    planned_actions: List[str] = field(default_factory=list)
    negotiation_strategy: str = "competitive"
    bidding_strategy: str = "cost_based"

@dataclass
class Agent:
    """Base class for all agents in the multi-agent system."""
    id: str
    agent_type: AgentType
    beliefs: AgentBeliefs = field(default_factory=AgentBeliefs)
    desires: AgentDesires = field(default_factory=AgentDesires)
    intentions: AgentIntentions = field(default_factory=AgentIntentions)
    
    # Agent state
    position: Tuple[int, int] = (1, 1)
    battery_level: float = 100.0
    status: str = "idle"  # 'idle', 'moving', 'loading', 'unloading', 'bidding', 'negotiating'
    capabilities: Dict[str, float] = field(default_factory=dict)
    
    # Communication
    message_queue: List[Message] = field(default_factory=list)
    known_agents: List[str] = field(default_factory=list)
    
    # Performance tracking
    tasks_completed: int = 0
    total_utility: float = 0.0
    communication_cost: float = 0.0
    
    def __post_init__(self):
        """Initialize agent capabilities."""
        if self.agent_type == AgentType.CRANE:
            self.capabilities = {
                'speed': 2.0,  # cells per second
                'capacity': 1.0,
                'battery_efficiency': 0.95,
                'accuracy': 0.98
            }
        elif self.agent_type == AgentType.COORDINATOR:
            self.capabilities = {
                'coordination_range': 100.0,
                'processing_speed': 10.0,
                'communication_bandwidth': 1000.0
            }
    
    def update_beliefs(self, new_beliefs: Dict[str, Any]):
        """Update agent beliefs."""
        for key, value in new_beliefs.items():
            if hasattr(self.beliefs, key):
                setattr(self.beliefs, key, value)
    
    def calculate_utility(self, task: Task) -> float:
        """Calculate utility for a task."""
        utility = 0.0
        
        # Priority-based utility
        utility += task.priority * 10
        
        # Distance-based cost
        distance = max(abs(task.x - self.position[0]), abs(task.y - self.position[1]))
        utility -= distance * 2
        
        # Deadline urgency
        if task.deadline:
            time_remaining = task.deadline - time.time()
            if time_remaining < 300:  # Less than 5 minutes
                utility += 20
            elif time_remaining < 600:  # Less than 10 minutes
                utility += 10
        
        # Workload balance consideration
        current_load = len(self.intentions.current_tasks)
        if current_load > 2:
            utility -= 5 * current_load
        
        return utility
    
    def generate_bid(self, task: Task) -> Dict[str, Any]:
        """Generate bid for a task."""
        utility = self.calculate_utility(task)
        
        # Calculate cost based on distance and time
        distance = max(abs(task.x - self.position[0]), abs(task.y - self.position[1]))
        travel_time = distance / self.capabilities.get('speed', 1.0)
        operation_time = 3.0 if task.task_type == 'storage' else 2.0
        total_time = travel_time + operation_time
        
        # Calculate bid price
        base_price = total_time * 10
        utility_bonus = utility * 2
        
        bid_price = max(1.0, base_price - utility_bonus)
        
        return {
            'agent_id': self.id,
            'task_id': task.id,
            'price': bid_price,
            'utility': utility,
            'estimated_time': total_time,
            'confidence': 0.9,
            'timestamp': time.time()
        }
    
    def send_message(self, receiver: str, message_type: str, content: Dict[str, Any]):
        """Send message to another agent."""
        message = Message(
            sender=self.id,
            receiver=receiver,
            message_type=message_type,
            content=content,
            timestamp=time.time()
        )
        
        # Add to global message system (would be handled by communication layer)
        return message
    
    def receive_message(self, message: Message):
        """Receive and process message."""
        self.message_queue.append(message)
        self.communication_cost += 0.1  # Communication cost
    
    def process_messages(self):
        """Process all messages in queue."""
        while self.message_queue:
            message = self.message_queue.pop(0)
            self._handle_message(message)
    
    def _handle_message(self, message: Message):
        """Handle incoming message."""
        if message.message_type == 'task_announcement':
            task = message.content['task']
            self._handle_task_announcement(task)
        elif message.message_type == 'bid_request':
            self._handle_bid_request(message.content)
        elif message.message_type == 'award_notification':
            self._handle_award_notification(message.content)
        elif message.message_type == 'system_update':
            self._handle_system_update(message.content)
    
    def _handle_task_announcement(self, task: Task):
        """Handle task announcement from coordinator."""
        # Decide whether to bid based on current state
        if self.status == 'idle' and self.battery_level > 20:
            # Generate bid
            bid = self.generate_bid(task)
            
            # Send bid to coordinator
            coordinator_msg = self.send_message(
                receiver='COORDINATOR',
                message_type='bid_submission',
                content=bid
            )
            
            self.status = 'bidding'
    
    def _handle_bid_request(self, content: Dict[str, Any]):
        """Handle bid request for specific task."""
        task = content['task']
        bid = self.generate_bid(task)
        
        # Send bid response
        response = self.send_message(
            receiver=content['requester'],
            message_type='bid_response',
            content=bid
        )
    
    def _handle_award_notification(self, content: Dict[str, Any]):
        """Handle task award notification."""
        task = content['task']
        if content['winner'] == self.id:
            # Accept task
            self.intentions.current_tasks.append(task)
            task.assigned_agent = self.id
            self.status = 'moving'
            self.total_utility += content['utility']
        else:
            # Lost bid
            pass
    
    def _handle_system_update(self, content: Dict[str, Any]):
        """Handle system state update."""
        # Update beliefs about system state
        self.update_beliefs(content)
    
    def execute_task(self, task: Task):
        """Execute assigned task."""
        # Move to task location
        target_pos = (task.x, task.y)
        distance = max(abs(target_pos[0] - self.position[0]), abs(target_pos[1] - self.position[1]))
        
        # Simulate movement
        travel_time = distance / self.capabilities.get('speed', 1.0)
        self.position = target_pos
        
        # Update battery
        battery_consumption = distance * 0.1
        self.battery_level = max(0, self.battery_level - battery_consumption)
        
        # Perform operation
        operation_time = 3.0 if task.task_type == 'storage' else 2.0
        self.status = 'loading' if task.task_type == 'storage' else 'unloading'
        
        # Complete task
        task.completed_time = time.time()
        self.tasks_completed += 1
        self.intentions.current_tasks.remove(task)
        self.status = 'idle'
        
        # Return to depot
        self.position = (1, 1)
        
        return travel_time + operation_time

In [ ]:
class CoordinatorAgent(Agent):
    """Coordinator agent for managing multi-agent system."""
    
    def __init__(self, agent_id: str = "COORDINATOR"):
        super().__init__(agent_id, AgentType.COORDINATOR)
        
        # Coordinator-specific attributes
        self.active_tasks: List[Task] = field(default_factory=list)
        self.task_market: Dict[str, List[Dict[str, Any]]] = field(default_factory=dict)
        self.auction_history: List[Dict[str, Any]] = field(default_factory=list)
        self.agent_registry: Dict[str, Agent] = field(default_factory=dict)
        
        # Market parameters
        self.auction_duration: float = 5.0  # seconds
        self.min_bids: int = 1
        self.market_fee: float = 0.05  # 5% fee
        
    def register_agent(self, agent: Agent):
        """Register an agent with the coordinator."""
        self.agent_registry[agent.id] = agent
        self.known_agents.append(agent.id)
        
        # Initialize agent beliefs
        agent.beliefs.agent_capabilities[agent.id] = agent.capabilities
        
        # Send welcome message
        welcome_msg = self.send_message(
            receiver=agent.id,
            message_type='welcome',
            content={'coordinator_id': self.id, 'system_time': time.time()}
        )
        
        # Send agent capabilities to all agents
        self._broadcast_agent_capabilities()
    
    def _broadcast_agent_capabilities(self):
        """Broadcast agent capabilities to all agents."""
        capabilities = {
            agent_id: agent.capabilities 
            for agent_id, agent in self.agent_registry.items()
        }
        
        for agent_id in self.agent_registry:
            if agent_id != self.id:
                self.send_message(
                    receiver=agent_id,
                    message_type='capabilities_update',
                    content={'capabilities': capabilities}
                )
    
    def announce_task(self, task: Task):
        """Announce new task to all agents."""
        self.active_tasks.append(task)
        
        # Create task market
        self.task_market[task.id] = []
        
        # Announce to all agents
        for agent_id in self.agent_registry:
            if agent_id != self.id and self.agent_registry[agent_id].agent_type == AgentType.CRANE:
                self.send_message(
                    receiver=agent_id,
                    message_type='task_announcement',
                    content={'task': task, 'deadline': time.time() + self.auction_duration}
                )
        
        # Schedule auction
        self._schedule_task_auction(task)
    
    def _schedule_task_auction(self, task: Task):
        """Schedule auction for task allocation."""
        print(f"\n[COORDINATOR] Starting auction for task {task.id}")
        print(f"[COORDINATOR] Task type: {task.task_type}, Location: ({task.x}, {task.y}), Priority: {task.priority}")
        
        # Wait for bids
        time.sleep(self.auction_duration)
        
        # Evaluate bids
        bids = self.task_market.get(task.id, [])
        
        if len(bids) >= self.min_bids:
            # Select winner
            winner = self._select_auction_winner(bids, task)
            
            # Award task
            self._award_task(task, winner)
        else:
            # No bids, re-announce or handle differently
            print(f"[COORDINATOR] No bids received for task {task.id}, re-announcing...")
            self.announce_task(task)
    
    def _select_auction_winner(self, bids: List[Dict[str, Any]], task: Task) -> str:
        """Select winner from bids."""
        # Sort by price (lowest wins)
        sorted_bids = sorted(bids, key=lambda b: b['price'])
        
        winner = sorted_bids[0]['agent_id']
        winning_price = sorted_bids[0]['price']
        
        print(f"[COORDINATOR] Auction winner: {winner} with price {winning_price:.2f}")
        
        # Record auction result
        self.auction_history.append({
            'task_id': task.id,
            'winner': winner,
            'winning_price': winning_price,
            'num_bids': len(bids),
            'timestamp': time.time()
        })
        
        return winner
    
    def _award_task(self, task: Task, winner: str):
        """Award task to winning agent."""
        winner_agent = self.agent_registry.get(winner)
        
        if winner_agent:
            # Calculate market fee
            winning_bid = next((b for b in self.task_market[task.id] if b['agent_id'] == winner), None)
            if winning_bid:
                market_fee = winning_bid['price'] * self.market_fee
            else:
                market_fee = 0
            
            # Send award notification
            award_msg = self.send_message(
                receiver=winner,
                message_type='award_notification',
                content={
                    'task': task,
                    'winner': winner,
                    'price': winning_bid['price'] if winning_bid else 0,
                    'utility': winning_bid['utility'] if winning_bid else 0,
                    'market_fee': market_fee
                }
            )
            
            # Update market
            del self.task_market[task.id]
            if task in self.active_tasks:
                self.active_tasks.remove(task)
            
            print(f"[COORDINATOR] Task {task.id} awarded to {winner}")
        else:
            print(f"[COORDINATOR] Error: Winner agent {winner} not found")
    
    def receive_bid(self, bid: Dict[str, Any]):
        """Receive bid from agent."""
        task_id = bid['task_id']
        
        if task_id in self.task_market:
            self.task_market[task_id].append(bid)
            print(f"[COORDINATOR] Received bid from {bid['agent_id']} for task {task_id}: {bid['price']:.2f}")
        else:
            print(f"[COORDINATOR] Warning: Bid received for unknown task {task_id}")
    
    def update_system_state(self):
        """Update system state and broadcast to agents."""
        # Calculate system metrics
        total_agents = len(self.agent_registry)
        active_cranes = sum(1 for agent in self.agent_registry.values() 
                           if agent.agent_type == AgentType.CRANE and agent.status != 'idle')
        
        pending_tasks = len(self.active_tasks)
        completed_tasks = sum(agent.tasks_completed for agent in self.agent_registry.values())
        
        # Calculate system load
        system_load = active_cranes / max(1, total_agents)
        
        # Update market prices based on load
        market_price_multiplier = 1.0 + system_load * 0.5
        
        # Broadcast system update
        system_update = {
            'system_load': system_load,
            'active_cranes': active_cranes,
            'pending_tasks': pending_tasks,
            'completed_tasks': completed_tasks,
            'market_price_multiplier': market_price_multiplier,
            'timestamp': time.time()
        }
        
        for agent_id in self.agent_registry:
            if agent_id != self.id:
                self.send_message(
                    receiver=agent_id,
                    message_type='system_update',
                    content=system_update
                )
    
    def get_system_performance(self) -> Dict[str, Any]:
        """Get system performance metrics."""
        total_tasks = sum(agent.tasks_completed for agent in self.agent_registry.values())
        total_utility = sum(agent.total_utility for agent in self.agent_registry.values())
        total_communication_cost = sum(agent.communication_cost for agent in self.agent_registry.values())
        
        # Calculate average completion time
        completion_times = []
        for agent in self.agent_registry.values():
            if hasattr(agent, 'task_completion_times'):
                completion_times.extend(agent.task_completion_times)
        
        avg_completion_time = np.mean(completion_times) if completion_times else 0
        
        return {
            'total_tasks_completed': total_tasks,
            'total_utility': total_utility,
            'communication_overhead': total_communication_cost,
            'avg_completion_time': avg_completion_time,
            'num_auctions': len(self.auction_history),
            'active_agents': len(self.agent_registry),
            'system_efficiency': total_tasks / max(1, total_tasks + len(self.active_tasks))
        }

In [ ]:
class MultiAgentASRS:
    """Multi-Agent System for AS/RS Task Interleaving."""
    
    def __init__(self, num_crane=3, num_aisles=6, aisle_length=20, aisle_height=10):
        """Initialize multi-agent system."""
        self.num_crane = num_crane
        self.num_aisles = num_aisles
        self.aisle_length = aisle_length
        self.aisle_height = aisle_height
        
        # Initialize coordinator
        self.coordinator = CoordinatorAgent()
        
        # Initialize agents
        self.agents = {}
        self._init_crane_agents()
        
        # Communication system
        self.message_router = {}
        
        # Task management
        self.task_queue = []
        self.completed_tasks = []
        
        # Performance tracking
        self.system_metrics = []
        self.auction_results = []
        
    def _init_crane_agents(self):
        """Initialize crane agents."""
        for i in range(self.num_crane):
            agent_id = f"CRANE_{i+1}"
            crane_agent = Agent(
                id=agent_id,
                agent_type=AgentType.CRANE,
                position=(1, 1),
                capabilities={
                    'speed': 2.0,
                    'capacity': 1.0,
                    'battery_efficiency': 0.95,
                    'accuracy': 0.98
                }
            )
            
            # Register with coordinator
            self.coordinator.register_agent(crane_agent)
            self.agents[agent_id] = crane_agent
            
            # Set up message routing
            self.message_router[agent_id] = crane_agent
            self.message_router[self.coordinator.id] = self.coordinator
    
    def add_task(self, task_id: str, task_type: str, x: int, y: int, priority: int, deadline: Optional[float] = None):
        """Add new task to the system."""
        task = Task(
            id=task_id,
            task_type=task_type,
            x=x,
            y=y,
            priority=priority,
            deadline=deadline,
            created_time=time.time()
        )
        
        self.task_queue.append(task)
        
        # Announce task through coordinator
        self.coordinator.announce_task(task)
        
        print(f"[SYSTEM] Task {task_id} added to queue: {task_type} at ({x}, {y}), priority {priority}")
    
    def route_message(self, message: Message):
        """Route message to appropriate agent."""
        receiver = self.message_router.get(message.receiver)
        
        if receiver:
            receiver.receive_message(message)
        else:
            print(f"[SYSTEM] Warning: Unknown receiver {message.receiver}")
    
    def process_bid(self, bid: Dict[str, Any]):
        """Process bid submission."""
        self.coordinator.receive_bid(bid)
        
    def step(self, dt: float = 1.0):
        """Advance simulation by dt seconds."""
        # Process messages
        for agent in self.agents.values():
            agent.process_messages()
        
        # Process coordinator messages
        self.coordinator.process_messages()
        
        # Execute tasks
        for agent in self.agents.values():
            if agent.intentions.current_tasks:
                task = agent.intentions.current_tasks[0]
                execution_time = agent.execute_task(task)
                
                # Record completion
                self.completed_tasks.append(task)
                
                # Track performance
                if not hasattr(agent, 'task_completion_times'):
                    agent.task_completion_times = []
                agent.task_completion_times.append(execution_time)
        
        # Update system state
        self.coordinator.update_system_state()
        
        # Record metrics
        metrics = self.coordinator.get_system_performance()
        self.system_metrics.append(metrics)
        
    def run_simulation(self, duration_hours: float = 2.0, task_rate: float = 1.0):
        """Run complete multi-agent simulation."""
        print("=" * 60)
        print(f"MULTI-AGENT AS/RS SIMULATION")
        print(f"=" * 60)
        print(f"Duration: {duration_hours} hours")
        print(f"Task rate: {task_rate} tasks/minute")
        print(f"Crane agents: {self.num_crane}")
        print()
        
        # Simulation parameters
        total_duration = duration_hours * 3600  # Convert to seconds
        task_interval = 60.0 / task_rate  # Seconds between tasks
        next_task_time = 0.0
        task_counter = 0
        
        start_time = time.time()
        current_time = 0.0
        
        print("Starting simulation...")
        
        while current_time < total_duration:
            # Add tasks at specified rate
            if current_time >= next_task_time:
                task_counter += 1
                task_id = f"TASK_{task_counter}"
                task_type = random.choice(['storage', 'retrieval'])
                x = random.randint(1, self.aisle_length)
                y = random.randint(1, self.aisle_height)
                priority = random.randint(1, 10)
                
                self.add_task(task_id, task_type, x, y, priority)
                next_task_time += task_interval
            
            # Step simulation
            self.step(dt=dt)
            current_time += dt
            
            # Progress reporting
            if int(current_time) % 300 == 0:  # Every 5 minutes
                completed = len(self.completed_tasks)
                pending = len(self.coordinator.active_tasks)
                active_agents = sum(1 for agent in self.agents.values() if agent.status != 'idle')
                
                print(f"Time: {current_time/3600:.1f}h | Tasks: {completed}/{completed+pending} | "
                      f"Active Agents: {active_agents}/{self.num_crane}")
        
        end_time = time.time()
        simulation_time = end_time - start_time
        
        print(f"\nSimulation completed in {simulation_time:.2f} seconds")
        
        # Final results
        final_metrics = self.coordinator.get_system_performance()
        
        print(f"\nFinal Results:")
        print(f"Total tasks completed: {final_metrics['total_tasks_completed']}")
        print(f"System efficiency: {final_metrics['system_efficiency']:.1%}")
        print(f"Average completion time: {final_metrics['avg_completion_time']:.2f} seconds")
        print(f"Communication overhead: {final_metrics['communication_overhead']:.2f}")
        print(f"Total auctions: {final_metrics['num_auctions']}")
        
        return final_metrics
    
    def analyze_agent_performance(self):
        """Analyze individual agent performance."""
        print("\n" + "=" * 60)
        print("AGENT PERFORMANCE ANALYSIS")
        print("=" * 60)
        
        for agent_id, agent in self.agents.items():
            print(f"\n{agent_id}:")
            print(f"  Tasks completed: {agent.tasks_completed}")
            print(f"  Total utility: {agent.total_utility:.2f}")
            print(f"  Communication cost: {agent.communication_cost:.2f}")
            print(f"  Current status: {agent.status}")
            print(f"  Battery level: {agent.battery_level:.1f}%")
            print(f"  Position: {agent.position}")
            
            if hasattr(agent, 'task_completion_times') and agent.task_completion_times:
                print(f"  Avg completion time: {np.mean(agent.task_completion_times):.2f}s")
                print(f"  Completion time std: {np.std(agent.task_completion_times):.2f}s")
            
            # Agent efficiency
            if agent.tasks_completed > 0:
                efficiency = agent.total_utility / agent.tasks_completed
                print(f"  Efficiency: {efficiency:.2f} utility/task")

In [ ]:
try:
    # Initialize the Multi-Agent System
    print("AS/RS Task Interleaving - Multi-Agent System")
    print(f"Number of crane agents: 3")
    print(f"AS/RS dimensions: {6} aisles x {20} x {10}")
    print()
    
    # Create multi-agent system
    mas_system = MultiAgentASRS(num_crane=3, num_aisles=6, aisle_length=20, aisle_height=10)
    
    print("Multi-Agent System initialized successfully!")
    print(f"Active agents: {len(mas_system.agents)}")
    print(f"Coordinator: {mas_system.coordinator.id}")
    print(f"Total agents in system: {len(mas_system.agents) + 1}")
    print(f"Message routing: {len(mas_system.message_router)} routes")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: 'Field' object does not support item assignment...]

In [ ]:
try:
    # Run baseline simulation
    print("\n" + "=" * 60)
    print("BASELINE SIMULATION")
    print("=" * 60)
    
    baseline_results = mas_system.run_simulation(
        duration_hours=2.0,
        task_rate=1.0
    )
    
    # Analyze agent performance
    mas_system.analyze_agent_performance()
    
    # Store baseline results
    baseline_metrics = baseline_results.copy()
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'mas_system' is not defined...]

In [ ]:
try:
    # Visualize comprehensive results
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # Plot 1: Task completion over time
    time_points = [i * 60 for i in range(len(mas_system.system_metrics))]
    completed_tasks = [m['total_tasks_completed'] for m in mas_system.system_metrics]
    system_efficiency = [m['system_efficiency'] for m in mas_system.system_metrics]
    
    ax1_twin = ax1.twinx()
    line1 = ax1.plot(time_points, completed_tasks, 'b-', linewidth=2, label='Tasks Completed')
    line2 = ax1_twin.plot(time_points, [e * 100 for e in system_efficiency], 'r--', linewidth=2, label='System Efficiency (%)')
    
    ax1.set_xlabel('Time (seconds)')
    ax1.set_ylabel('Tasks Completed')
    ax1.set_title('Multi-Agent System Performance Over Time')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1_twin.set_ylabel('System Efficiency (%)')
    
    # Plot 2: Agent performance comparison
    agent_ids = list(mas_system.agents.keys())
    agent_tasks = [mas_system.agents[aid].tasks_completed for aid in agent_ids]
    agent_utilities = [mas_system.agents[aid].total_utility for aid in agent_ids]
    agent_costs = [mas_system.agents[aid].communication_cost for aid in agent_ids]
    
    x = np.arange(len(agent_ids))
    width = 0.25
    
    bars2_1 = ax2.bar(x - width, agent_tasks, width, label='Tasks', alpha=0.7)
    bars2_2 = ax2.bar(x, [u/10 for u in agent_utilities], width, label='Utility/10', alpha=0.7)
    bars2_3 = ax2.bar(x + width, agent_costs, width, label='Comm Cost', alpha=0.7)
    
    ax2.set_xlabel('Agents')
    ax2.set_ylabel('Count / Value')
    ax2.set_title('Agent Performance Metrics')
    ax2.set_xticks(x)
    ax2.set_xticklabels(agent_ids)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Communication overhead analysis
    comm_overheads = [m['communication_overhead'] for m in mas_system.system_metrics]
    num_auctions = [m['num_auctions'] for m in mas_system.system_metrics]
    
    ax3.plot(time_points, comm_overheads, 'g-', linewidth=2, label='Communication Overhead')
    ax3_twin = ax3.twinx()
    ax3_twin.plot(time_points, num_auctions, 'orange', linewidth=2, label='Number of Auctions')
    
    ax3.set_xlabel('Time (seconds)')
    ax3.set_ylabel('Communication Overhead')
    ax3.set_title('Communication and Auction Analysis')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    ax3_twin.set_ylabel('Number of Auctions')
    
    # Plot 4: System efficiency and utility
    total_utility = [m['total_utility'] for m in mas_system.system_metrics]
    system_eff = [m['system_efficiency'] for m in mas_system.system_metrics]
    
    ax4.plot(time_points, total_utility, 'purple', linewidth=2, label='Total Utility')
    ax4_twin = ax4.twinx()
    ax4_twin.plot(time_points, [e * 100 for e in system_eff], 'brown', linewidth=2, label='Efficiency (%)')
    
    ax4.set_xlabel('Time (seconds)')
    ax4.set_ylabel('Total Utility')
    ax4.set_title('System Utility and Efficiency')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    ax4_twin.set_ylabel('Efficiency (%)')
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print(f"\nMulti-Agent System Performance Summary:")
    print(f"Total tasks completed: {baseline_metrics['total_tasks_completed']}")
    print(f"System efficiency: {baseline_metrics['system_efficiency']:.1%}")
    print(f"Average completion time: {baseline_metrics['avg_completion_time']:.2f} seconds")
    print(f"Communication overhead: {baseline_metrics['communication_overhead']:.2f}")
    print(f"Total auctions: {baseline_metrics['num_auctions']}")
    print(f"Active agents: {len(mas_system.agents)}")
    
    # Performance analysis
    print(f"\nPerformance Analysis:")
    print(f"• Multi-agent system demonstrates efficient task allocation")
    print(f"• Communication overhead remains manageable")
    print(f"• Auction mechanism enables fair resource distribution")
    print(f"• System efficiency: {baseline_metrics['system_efficiency']:.1%}")
    print(f"• Distributed decision-making emerges from local interactions")
    
    # Agent analysis
    print(f"\nAgent Analysis:")
    for agent_id, agent in mas_system.agents.items():
        if agent.tasks_completed > 0:
            efficiency = agent.total_utility / agent.tasks_completed
            print(f"{agent_id}: {agent.tasks_completed} tasks, {efficiency:.2f} utility/task")
        else:
            print(f"{agent_id}: No tasks completed")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'mas_system' is not defined...]

### Why This Tier Exists vs Earlier Tiers

**Tier 6: Multi-Agent System** provides distributed intelligence with:
- **Autonomous decision-making** through independent agent reasoning
- **Emergent behavior** from local interactions without central control
- **Market-based coordination** using Contract Net Protocol and dynamic pricing
- **Self-organization** adapting to changing conditions and agent failures

### Pros / Cons vs Alternative Approaches

**Advantages:**
- ✅ **Distributed intelligence** - No single point of failure
- ✅ **Scalability** - Easy to add/remove agents dynamically
- ✅ **Adaptability** - System self-organizes around changes
- ✅ **Resilience** - Continues operating despite agent failures
- ✅ **Local decision-making** - Faster responses without central coordination

**Disadvantages:**
- ❌ **Communication overhead** - Message passing adds computational cost
- ❌ **Coordination complexity** - Managing agent interactions is challenging
- ❌ **No global optimality guarantee** - Emergent behavior may be suboptimal
- ❌ **Message routing complexity** - Requires robust communication infrastructure
- ❌ **Debugging complexity** - Hard to trace system-wide issues

### When to Use This Tier

**Use Multi-Agent System when:**
- System needs to be highly resilient to failures
- Scalability is required (adding/removing agents dynamically)
- Distributed decision-making is preferred over centralized control
- Multiple stakeholders need autonomous decision-making capabilities
- Real-time adaptation to changing conditions is critical
- System spans multiple domains requiring local expertise

**Consider other tiers when:**
- Global optimization guarantees are required (use Tier 1)
- Computational efficiency is critical (use Tier 2 or 3)
- Centralized control is acceptable (use Tier 4 or 5)
- Simple coordination problems with few agents (use simpler approaches)
- Communication infrastructure is limited or unreliable